In [370]:
import pandas as pd
from data_profiling import ProfileReport

In [371]:
# Loading a sample Grandprix - 2025 Australian Grand Prix
sample_loc = "f1_parquet_data/2025/Round_1_Australian_Grand_Prix/R"
tele_df = pd.read_parquet(f"{sample_loc}/telemetry.parquet")
lap_df = pd.read_parquet(f"{sample_loc}/laps.parquet")
result_df = pd.read_parquet(f"{sample_loc}/results.parquet")
weather_df = pd.read_parquet(f"{sample_loc}/weather.parquet")

### **Step 1: Data Profiling**

In [372]:
# import os
# import sys
# import contextlib

# # 1. List your 4 parquet files
# parquet_files = {
#     "Lap Data": lap_df,
#     "Telemetry Data": tele_df,
#     "Results Data": result_df,
#     "Weather Data": weather_df
# }
# reports_html = {}
# # 2. Loop through files, load data, and generate profiling HTML snippets
# for title, filepath in parquet_files.items():
#     with open(os.devnull, "w") as f, contextlib.redirect_stdout(f):
#         print(f"Profiling {title} ({filepath})...")
#         # Generate profile report
#         profile = ProfileReport(filepath, title=title, explorative=True, progress_bar=False)
#         # Export report as an HTML string (as_html=True avoids saving 4 separate files)
#         reports_html[title] = profile.to_html()

# # 3. Create a master HTML file with tab navigation
# master_html = """
# <!DOCTYPE html>
# <html>
# <head>
#     <meta charset="UTF-8">
#     <title>Combined Parquet Data Profiles</title>
#     <style>
#         body { font-family: Arial, sans-serif; margin: 0; padding: 20px; background-color: #f4f4f9; }
#         h1 { text-align: center; color: #333; }
#         /* Tab container styling */
#         .tab { overflow: hidden; border-bottom: 1px solid #ccc; background-color: #ffffff; border-radius: 5px 5px 0 0; }
#         /* Style the buttons inside the tab */
#         .tab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 20px; transition: 0.3s; font-size: 16px; font-weight: bold; color: #555; }
#         /* Change background color of buttons on hover */
#         .tab button:hover { background-color: #ddd; color: #000; }
#         /* Create an active/current tablink class */
#         .tab button.active { background-color: #007bff; color: white; }
#         /* Style the tab content */
#         .tabcontent { display: none; padding: 6px 12px; background-color: #ffffff; border-top: none; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
#         iframe { width: 100%; height: 800px; border: none; }
#     </style>
# </head>
# <body>

# <h1>Multi-Dataset Profiling Report</h1>

# <div class="tab">
# """

# # Generate tab buttons dynamically
# first = True
# for title in reports_html.keys():
#     active_class = "active" if first else ""
#     safe_id = title.replace(" ", "_")
#     master_html += f'  <button class="tablinks {active_class}" onclick="openTab(event, \'{safe_id}\')">{title}</button>\n'
#     first = False

# master_html += "</div>\n"

# # Generate tab content sections using iframes containing the HTML strings
# first = True
# for title, html_content in reports_html.items():
#     display_style = "block" if first else "none"
#     safe_id = title.replace(" ", "_")

#     # We use srcdoc to inject the large profiling HTML snippet safely into an iframe container
#     master_html += f"""
# <div id="{safe_id}" class="tabcontent" style="display: {display_style};">
#     <iframe srcdoc="{html_content.replace('"', '&quot;')}"></iframe>
# </div>
# """
#     first = False

# # 4. JavaScript for tab switching interactivity
# master_html += """
# <script>
# function openTab(evt, tabName) {
#     var i, tabcontent, tablinks;
#     tabcontent = document.getElementsByClassName("tabcontent");
#     for (i = 0; i < tabcontent.length; i++) {
#         tabcontent[i].style.display = "none";
#     }
#     tablinks = document.getElementsByClassName("tablinks");
#     for (i = 0; i < tablinks.length; i++) {
#         tablinks[i].className = tablinks[i].className.replace(" active", "");
#     }
#     document.getElementById(tabName).style.display = "block";
#     evt.currentTarget.className += " active";
# }
# </script>

# </body>
# </html>
# """

# # 5. Save the final combined report
# output_filename = "combined_profile_report.html"
# with open(output_filename, "w", encoding="utf-8") as f:
#     f.write(master_html)

# print(f"Successfully generated tabbed report: {output_filename}")

### **Step 2: Cleaning each Dataset Type**

In this step, we will perform data cleaning for the 4 tables seperately

#### **Lap Data**

In [373]:
print(lap_df.columns.to_list())
print("Number of rows: ", lap_df.shape[0])
print("Number of columns: ", lap_df.shape[1])

['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']
Number of rows:  927
Number of columns:  31


In [374]:
# Drop unnecessary columns
lap_drop = ["Time", "DriverNumber", "Sector1SessionTime","Sector2SessionTime","Sector3SessionTime","SpeedI1","SpeedI2","SpeedFL","SpeedST","LapStartTime","LapStartDate","FastF1Generated"]
lap_df = lap_df.drop(lap_drop, axis=1)
lap_df.shape

(927, 19)

In [375]:
# 1. Drop Duplicates
lap_df.duplicated().sum()
lap_df = lap_df.drop_duplicates()

In [376]:
# 2. Standardize Driver Names
driver_map = dict(zip(result_df["Abbreviation"], result_df["FirstName"]+" "+result_df["LastName"]))
lap_df["Driver"] = lap_df["Driver"].map(driver_map).fillna(lap_df["Driver"])

In [377]:
# 3. Convert Lap Times into numeric seconds
columnToNumeric = ["LapTime", "Sector1Time", "Sector2Time","Sector3Time"]
for i in columnToNumeric:
    lap_df[i] = lap_df[i].dt.total_seconds()

In [378]:
# 4. Handling missing sector times
# Missing Sector Times can be due to PitLaps, Formation laps or retirements.
lap_df["MissingSector1"] = lap_df["Sector1Time"].isna()
lap_df["MissingSector2"] = lap_df["Sector2Time"].isna()
lap_df["MissingSector3"] = lap_df["Sector3Time"].isna()

# 5. Remove the formation laps
lap_df = lap_df[lap_df["LapNumber"] != 0]

In [379]:
# 6. Handling In-laps and Out-laps
lap_df["IsInLap"] = lap_df["PitInTime"].isna()
lap_df["IsOutLap"] = lap_df["PitOutTime"].isna()

In [380]:
# 7. Remove slow laps. For example, laps due to Safety Car, Virtual Safety Car or Red Flag
# Track Status can be used for it.
# 1: Track clear / Normal conditions
# 2: Yellow flag (hazard on or near the track)
# 4: Safety Car (SC) deployed
# 5: Red flag (session suspended)
# 6: Virtual Safety Car (VSC) deployed
# 7: VSC ending (preparing to go green)

# lap_df["TrackStatus"].to_string()
trackStatus_map = {
    "1": "Track Clear", "2": "Yellow Flag",
    "4": "Safety Car deployed", "5": "Red Flag (Session Suspended)",
    "6": "Virtual Safety Car deployed","7": "Virtual Safety Car ending"
}
status_list = []
for i in lap_df["TrackStatus"]:
    status = ""
    for j in i:
        status += trackStatus_map[j] + "-"
    status_list.append(status)

lap_df["TrackData"] = status_list
lap_df["TrackData"] = lap_df["TrackData"].str.rstrip("-")
lap_df["TrackData"].unique()

array(['Track Clear-Yellow Flag-Safety Car deployed',
       'Safety Car deployed', 'Safety Car deployed-Track Clear',
       'Track Clear', 'Track Clear-Yellow Flag',
       'Yellow Flag-Safety Car deployed', 'Yellow Flag-Track Clear'],
      dtype=object)

In [381]:
# 8. Tyre Compound Standardization and Encoding
compound_df = {
    "SOFT": 1,
    "MEDIUM": 2,
    "HARD": 3,
    "INTERMEDIATE": 4,
    "WET": 5
}
lap_df["Compound"] = lap_df["Compound"].map(compound_df).fillna(lap_df["Compound"])
lap_df["Compound"].unique()

array([4, 2, 3], dtype=int64)

In [382]:
# Fixing the dtypes of the columns
lap_df["LapNumber"] = lap_df["LapNumber"].astype('int64')
lap_df["Stint"] = lap_df["Stint"].astype('int64')
lap_df["IsPersonalBest"] = lap_df["IsPersonalBest"].astype(bool)
lap_df['TyreLife'] = lap_df['TyreLife'].astype(bool)
lap_df['TrackStatus'] = lap_df["TrackStatus"].astype('int64')

#### **Telemetry Data**

In [383]:
print(tele_df.columns.to_list())
print(tele_df.shape[0], tele_df.shape[1])

['Date', 'SessionTime', 'DriverAhead', 'DistanceToDriverAhead', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Distance', 'RelativeDistance', 'Status', 'X', 'Y', 'Z', 'Driver', 'LapNumber']
754313 20


In [384]:
# 1. Check for any duplicates. If exist, remove them
tele_df.duplicated().sum()
# tele_df.drop_duplicates()
# tele_df.shape

0

In [385]:
# 2. Standardization
tele_df["Driver"] = tele_df["Driver"].map(driver_map).fillna(tele_df["Driver"])
tele_df["RPM"] = tele_df["RPM"].astype(int)

# Speed is in Km/h and distance is in meters

In [386]:
tele_df.head()

,Date,SessionTime,DriverAhead,DistanceToDriverAhead,Time,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Distance,RelativeDistance,Status,X,Y,Z,Driver,LapNumber
0,2025-03-16 04:18:22.974,0 days 01:11:00.355000,,0.2,0 days 00:00:00,10114,0.0,2,16.0,True,1,interpolation,-0.001617,-3.055566e-07,OnTrack,-941.083631,-1575.858789,86.0,Max Verstappen,1.0
1,2025-03-16 04:18:23.093,0 days 01:11:00.474000,,0.2,0 days 00:00:00.119000,10043,0.0,2,16.0,True,1,car,0.000000,0.000000e+00,OnTrack,-941.063519,-1575.892755,86.0,Max Verstappen,1.0
2,2025-03-16 04:18:23.168,0 days 01:11:00.549000,,0.2,0 days 00:00:00.194000,9964,0.0,2,16.0,True,1,pos,0.003298,6.230908e-07,OnTrack,-941.000000,-1576.000000,86.0,Max Verstappen,1.0
3,2025-03-16 04:18:23.372,0 days 01:11:00.753000,,0.2,0 days 00:00:00.398000,9752,0.0,2,16.0,True,1,car,0.000000,0.000000e+00,OnTrack,-940.635475,-1576.615464,86.0,Max Verstappen,1.0
4,2025-03-16 04:18:23.548,0 days 01:11:00.929000,,0.2,0 days 00:00:00.574000,8520,0.0,2,16.0,True,1,pos,-0.040207,-7.595834e-06,OnTrack,-941.000000,-1576.000000,86.0,Max Verstappen,1.0


In [387]:
# Handle missing values
tele_drop = ['DistanceToDriverAhead','RelativeDistance']
tele_df = tele_df.drop(tele_drop, axis=1)
tele_df.isna().sum()

Date           0
SessionTime    0
DriverAhead    0
Time           0
RPM            0
Speed          0
nGear          0
Throttle       0
Brake          0
DRS            0
Source         0
Distance       0
Status         0
X              0
Y              0
Z              0
Driver         0
LapNumber      0
dtype: int64

In [388]:
tele_df['Source'].unique()

array(['interpolation', 'car', 'pos'], dtype=object)

In [389]:
# Removing Impossible values
# tele_df['Speed'] = tele_df[(tele_df['Speed'] >= 0) | (tele_df['Speed'] < 400)]
# tele_df['Throttle'] = tele_df[(tele_df['Throttle']>=0) | tele_df["Throttle"] <= 100]


In [390]:
# 5. Synchronizing Telemetry Frequency.
tele_df['Date'] = pd.to_datetime(tele_df['Date'])
tele_df = tele_df.set_index('Date')
tele_df = tele_df[~tele_df.index.duplicated(keep='first')]

# Seperate columns by type to apply interpolation rules
# - Continous (Speed, throttle, RPM) need linear interpolation
# - Discrete (nGear, DRS, Brake) need forward fill
continuous_cols = ['Speed', 'RPM', 'Throttle', 'X', 'Y', 'Z']
discrete_cols = ['nGear', 'DRS', 'Brake']

existing_cont = [col for col in continuous_cols if col in tele_df.columns]
existing_disc = [col for col in discrete_cols if col in tele_df.columns]
rule = '100ms'
tele_df_resampled = pd.concat([
    tele_df[existing_cont].resample(rule).mean().interpolate(method='linear'),
    tele_df[existing_disc].resample(rule).ffill()
], axis=1)
tele_df_resampled = tele_df_resampled.reset_index()

tele_df_resampled.head()

,Date,Speed,RPM,Throttle,X,Y,Z,nGear,DRS,Brake
0,2025-03-16 04:18:22.900,0.0,10114.0,16.0,-941.083631,-1575.858789,86.0,NaN,NaN,NaN
1,2025-03-16 04:18:23.000,0.0,10043.0,16.0,-941.063519,-1575.892755,86.0,2.0,1.0,True
2,2025-03-16 04:18:23.100,0.0,9964.0,16.0,-941.000000,-1576.000000,86.0,2.0,1.0,True
3,2025-03-16 04:18:23.200,0.0,9858.0,16.0,-940.817737,-1576.307732,86.0,2.0,1.0,True
4,2025-03-16 04:18:23.300,0.0,9752.0,16.0,-940.635475,-1576.615464,86.0,2.0,1.0,True


In [391]:
import numpy as np
# 6. Remove corrupted GPS points
# Drop null values
tele_df = tele_df.dropna(subset=['X','Y'])

# Remove static and zero glitches
tele_df = tele_df[~((tele_df['X']==0) & (tele_df['Y']==0))]

# Filter out spatial jumps
tele_df['Delta_X'] = tele_df['X'].diff()
tele_df['Delta_Y'] = tele_df['Y'].diff()
tele_df["Distance_Step"] = np.sqrt(tele_df['Delta_X']**2 + tele_df['Delta_Y']**2)

# Calculate time difference b/w rows
if 'Date' in tele_df.columns:
    parsed_dates = pd.to_datetime(tele_df['Date'], errors='coerce')
    tele_df['Time_Step'] = parsed_dates.diff().dt.total_seconds()
    tele_df['Time_Step'] = tele_df['Time_Step'].fillna(0.1)
else:
    tele_df['Time_Step'] = 0.1

# Speed checks based on GPS:
# If distance jumped per time step implies > 400 km/h, drop it
max_plausible_speed = 400
tele_df['Implied_Speed'] = np.where(tele_df['Time_Step'] > 0, tele_df['Distance_Step']/tele_df['Time_Step'], 0)
tele_df = tele_df[tele_df['Implied_Speed'] <= max_plausible_speed]

tele_df = tele_df.drop(columns=['Delta_X', 'Delta_Y', 'Distance_Step', 'Time_Step', 'Implied_Speed'])
tele_df.tail()

,SessionTime,DriverAhead,Time,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Distance,Status,X,Y,Z,Driver,LapNumber
Date,,,,,,,,,,,,,,,,,
2025-03-16 06:01:05.528,0 days 02:53:42.909000,,0 days 00:01:26.212000,10602,232.500000,6,100.0,False,0,pos,4919.794317,OnTrack,1105.000000,-3563.000000,86.000000,Oliver Bearman,57.0
2025-03-16 06:01:06.288,0 days 02:53:43.669000,,0 days 00:01:26.972000,11306,248.000000,6,100.0,False,0,car,4971.148889,OnTrack,744.103998,-3209.539005,87.165730,Oliver Bearman,57.0
2025-03-16 06:01:06.687,0 days 02:53:44.068000,,0 days 00:01:27.371000,11595,255.000000,6,100.0,False,0,car,4999.411389,OnTrack,540.220915,-3009.441075,87.990973,Oliver Bearman,57.0
2025-03-16 06:01:07.808,0 days 02:53:45.189000,,0 days 00:01:28.492000,10771,271.000000,7,100.0,False,0,car,5081.949722,OnTrack,-85.633449,-2395.376956,90.004045,Oliver Bearman,57.0
2025-03-16 06:01:08.187,0 days 02:53:45.568000,,0 days 00:01:28.871000,10930,275.190001,7,100.0,False,0,pos,5110.795022,OnTrack,-281.000000,-2205.000000,91.000000,Oliver Bearman,57.0


In [392]:
# # Create consistent Timestamp
td_l = ["SessionTime", "Time"]
for i in td_l:
    td = pd.to_timedelta(tele_df[i])
    h = td.dt.components.hours
    m = td.dt.components.minutes
    s = td.dt.components.seconds + td.dt.components.milliseconds / 1000

    tele_df[i] = (
        h.astype(str).str.zfill(2) + ":" +
        m.astype(str).str.zfill(2) + ":" +
        s.astype(str).str.zfill(2)
    )
tele_df.tail()

,SessionTime,DriverAhead,Time,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Distance,Status,X,Y,Z,Driver,LapNumber
Date,,,,,,,,,,,,,,,,,
2025-03-16 06:01:05.528,02:53:42.909,,00:01:26.212,10602,232.500000,6,100.0,False,0,pos,4919.794317,OnTrack,1105.000000,-3563.000000,86.000000,Oliver Bearman,57.0
2025-03-16 06:01:06.288,02:53:43.669,,00:01:26.972,11306,248.000000,6,100.0,False,0,car,4971.148889,OnTrack,744.103998,-3209.539005,87.165730,Oliver Bearman,57.0
2025-03-16 06:01:06.687,02:53:44.068,,00:01:27.371,11595,255.000000,6,100.0,False,0,car,4999.411389,OnTrack,540.220915,-3009.441075,87.990973,Oliver Bearman,57.0
2025-03-16 06:01:07.808,02:53:45.189,,00:01:28.492,10771,271.000000,7,100.0,False,0,car,5081.949722,OnTrack,-85.633449,-2395.376956,90.004045,Oliver Bearman,57.0
2025-03-16 06:01:08.187,02:53:45.568,,00:01:28.871,10930,275.190001,7,100.0,False,0,pos,5110.795022,OnTrack,-281.000000,-2205.000000,91.000000,Oliver Bearman,57.0


#### **Weather Data**

In [393]:
weather_df.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:07.776000,15.8,91.0,1009.2,True,18.8,211,2.6
1,0 days 00:01:07.777000,15.7,92.0,1009.2,True,18.8,273,2.6
2,0 days 00:02:07.763000,15.7,92.0,1009.0,True,18.8,271,2.2
3,0 days 00:03:07.777000,15.7,90.0,1009.1,True,18.8,261,5.3
4,0 days 00:04:07.778000,15.7,90.0,1009.4,True,18.8,246,5.6


In [394]:
# Check the null values
weather_df.isna().sum()

Time             0
AirTemp          0
Humidity         0
Pressure         0
Rainfall         0
TrackTemp        0
WindDirection    0
WindSpeed        0
dtype: int64

In [395]:
# Forming datatime stamp for consistency
td = pd.to_timedelta(weather_df['Time'])
h = td.dt.components.hours
m = td.dt.components.minutes
s = td.dt.components.seconds
ms = td.dt.components.milliseconds

weather_df['Time'] = (
    h.astype(str).str.zfill(2) + ":" +
    m.astype(str).str.zfill(2) + ":" +
    s.astype(str).str.zfill(2) + "." +
    ms.astype(str).str.zfill(3)
)
weather_df.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,00:00:07.776,15.8,91.0,1009.2,True,18.8,211,2.6
1,00:01:07.777,15.7,92.0,1009.2,True,18.8,273,2.6
2,00:02:07.763,15.7,92.0,1009.0,True,18.8,271,2.2
3,00:03:07.777,15.7,90.0,1009.1,True,18.8,261,5.3
4,00:04:07.778,15.7,90.0,1009.4,True,18.8,246,5.6


In [396]:
# Check for the value ranges
print("Shape before: ", weather_df.shape)

weather_df = weather_df[(weather_df['AirTemp'] >= 10) | (weather_df['AirTemp'] <= 45)]
weather_df = weather_df[(weather_df['TrackTemp'] >= 15) | (weather_df['TrackTemp'] <= 60)]
weather_df = weather_df[(weather_df['Humidity'] >= 10) | (weather_df['Humidity']<= 100)]
weather_df = weather_df[(weather_df['WindSpeed'] >= 0) | (weather_df['WindSpeed'] <= 15)]
weather_df = weather_df[(weather_df['WindDirection'] >= 0) | (weather_df['WindDirection'] <= 360)]
print("Shape After: ", weather_df.shape)


Shape before:  (178, 8)
Shape After:  (178, 8)


In [397]:
# Aggregate Weather Flags
summary = {
    'avg_track_temp': weather_df['TrackTemp'].mean(),
    'max_track_temp': weather_df['TrackTemp'].max(),
    'min_track_temp': weather_df['TrackTemp'].min(),
    'avg_air_temp':   weather_df['AirTemp'].mean(),
    'max_air_temp':   weather_df['AirTemp'].max(),
    'avg_humidity':   weather_df['Humidity'].mean(),
    'total_rainfall': weather_df['Rainfall'].sum() # True/False converted to count
}

print(summary)

{'avg_track_temp': 18.942134831460674, 'max_track_temp': 19.4, 'min_track_temp': 18.3, 'avg_air_temp': 15.707865168539326, 'max_air_temp': 16.6, 'avg_humidity': 78.42134831460675, 'total_rainfall': 58}


#### **Race Results Data**

In [398]:
print(result_df.columns)
result_df.head()

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps'],
      dtype='object')


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
4,4,L NORRIS,NOR,norris,McLaren,FF8000,mclaren,Lando,Norris,Lando Norris,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:42:06.304000,Finished,25.0,57.0
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,2.0,2,3.0,NaT,NaT,NaT,0 days 00:00:00.895000,Finished,18.0,57.0
63,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,...,3.0,3,4.0,NaT,NaT,NaT,0 days 00:00:08.481000,Finished,15.0,57.0
12,12,A ANTONELLI,ANT,antonelli,Mercedes,27F4D2,mercedes,Andrea Kimi,Antonelli,Andrea Kimi Antonelli,...,4.0,4,16.0,NaT,NaT,NaT,0 days 00:00:10.135000,Finished,12.0,57.0
23,23,A ALBON,ALB,albon,Williams,64C4FF,williams,Alexander,Albon,Alexander Albon,...,5.0,5,6.0,NaT,NaT,NaT,0 days 00:00:12.773000,Finished,10.0,57.0


In [399]:
to_drop = ["FirstName", "BroadcastName", "Abbreviation", "HeadshotUrl", "CountryCode", "Q1","Q2", "Q3", "ClassifiedPosition"]
result_df = result_df.drop(to_drop, axis=1)
result_df.shape


(20, 13)

In [400]:
result_df["DriverId"] = result_df["LastName"].str.lower()
result_df = result_df.drop(["LastName"], axis=1)
result_df.head()

,DriverNumber,DriverId,TeamName,TeamColor,TeamId,FullName,Position,GridPosition,Time,Status,Points,Laps
4,4,norris,McLaren,FF8000,mclaren,Lando Norris,1.0,1.0,0 days 01:42:06.304000,Finished,25.0,57.0
1,1,verstappen,Red Bull Racing,3671C6,red_bull,Max Verstappen,2.0,3.0,0 days 00:00:00.895000,Finished,18.0,57.0
63,63,russell,Mercedes,27F4D2,mercedes,George Russell,3.0,4.0,0 days 00:00:08.481000,Finished,15.0,57.0
12,12,antonelli,Mercedes,27F4D2,mercedes,Andrea Kimi Antonelli,4.0,16.0,0 days 00:00:10.135000,Finished,12.0,57.0
23,23,albon,Williams,64C4FF,williams,Alexander Albon,5.0,6.0,0 days 00:00:12.773000,Finished,10.0,57.0


In [401]:
# Change the dtypes of the columns
result_df["DriverNumber"] = result_df['DriverNumber'].astype(int)
result_df['Position'] = result_df['Position'].astype(float)
result_df['GridPosition'] = result_df['GridPosition'].astype(float)
result_df['Points'] = result_df['Points'].astype(float)
result_df['Laps'] = result_df['Laps'].astype(float)

In [402]:
def formatted_times(row):
    td = pd.to_timedelta(row['Time'], errors='coerce')
    total_seconds = td.total_seconds()
    if pd.isna(total_seconds):
        return pd.NaT

    h = int(total_seconds//3600)
    m = int((total_seconds%3600) // 60)
    s = int(total_seconds % 60)
    ms = int(round((total_seconds % 1) * 1000))

    if row['Position'] == 1.0:
        return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"
    else:
        total_mins = int(total_seconds // 60)
        return f"+{total_mins:02d}:{s:02d}.{ms:03d}"

result_df['FormattedTime'] = result_df.apply(formatted_times, axis=1)

In [403]:
print(lap_df["Team"].unique())
print(result_df["TeamName"].unique())

['Red Bull Racing' 'Alpine' 'Mercedes' 'Aston Martin' 'Ferrari'
 'Racing Bulls' 'Williams' 'Kick Sauber' 'Haas F1 Team' 'McLaren']
['McLaren' 'Red Bull Racing' 'Mercedes' 'Williams' 'Aston Martin'
 'Kick Sauber' 'Ferrari' 'Alpine' 'Racing Bulls' 'Haas F1 Team']


In [404]:
# Checking for DNF, DNS and DSQ
dnfs = result_df[result_df['Status'].isin(['Retired', 'Accident', 'Collision', 'Engine', 'Gearbox', 'Power Unit', 'Suspension', 'Brakes', 'Overheating'])]
dns_drivers = result_df[result_df['Status'] == 'DNS']
dsq_drivers = result_df[result_df['Status'] == 'Disqualified']

In [405]:

point_system = {
    1:25, 2:18, 3:15, 4:12, 5:10, 6:8, 7:6, 8:4, 9:2, 10:1
}
point_system


{1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

### **Step 3: Cross Dataset Cleaning**

In [408]:
# result_df = result_df.rename(columns={'FullName': 'Driver'})
print(lap_df['Driver'].unique())
print(result_df['Driver'].unique())

['Max Verstappen' 'Pierre Gasly' 'Andrea Kimi Antonelli' 'Fernando Alonso'
 'Charles Leclerc' 'Lance Stroll' 'Yuki Tsunoda' 'Alexander Albon'
 'Nico Hulkenberg' 'Liam Lawson' 'Esteban Ocon' 'Lando Norris'
 'Lewis Hamilton' 'Gabriel Bortoleto' 'Carlos Sainz' 'Isack Hadjar'
 'George Russell' 'Jack Doohan' 'Oscar Piastri' 'Oliver Bearman']
['Lando Norris' 'Max Verstappen' 'George Russell' 'Andrea Kimi Antonelli'
 'Alexander Albon' 'Lance Stroll' 'Nico Hulkenberg' 'Charles Leclerc'
 'Oscar Piastri' 'Lewis Hamilton' 'Pierre Gasly' 'Yuki Tsunoda'
 'Esteban Ocon' 'Oliver Bearman' 'Liam Lawson' 'Gabriel Bortoleto'
 'Fernando Alonso' 'Carlos Sainz' 'Jack Doohan' 'Isack Hadjar']


In [412]:
print(tele_df['Driver'].unique())
print(lap_df['Driver'].unique())

['Max Verstappen' 'Pierre Gasly' 'Andrea Kimi Antonelli' 'Fernando Alonso'
 'Charles Leclerc' 'Lance Stroll' 'Yuki Tsunoda' 'Alexander Albon'
 'Nico Hulkenberg' 'Liam Lawson' 'Esteban Ocon' 'Lando Norris'
 'Lewis Hamilton' 'Gabriel Bortoleto' 'George Russell' 'Oscar Piastri'
 'Oliver Bearman']
['Max Verstappen' 'Pierre Gasly' 'Andrea Kimi Antonelli' 'Fernando Alonso'
 'Charles Leclerc' 'Lance Stroll' 'Yuki Tsunoda' 'Alexander Albon'
 'Nico Hulkenberg' 'Liam Lawson' 'Esteban Ocon' 'Lando Norris'
 'Lewis Hamilton' 'Gabriel Bortoleto' 'Carlos Sainz' 'Isack Hadjar'
 'George Russell' 'Jack Doohan' 'Oscar Piastri' 'Oliver Bearman']


In [414]:
result_df.head()

,DriverNumber,DriverId,TeamName,TeamColor,TeamId,Driver,Position,GridPosition,Time,Status,Points,Laps,FormattedTime
4,4,norris,McLaren,FF8000,mclaren,Lando Norris,1.0,1.0,0 days 01:42:06.304000,Finished,25.0,57.0,01:42:06.304
1,1,verstappen,Red Bull Racing,3671C6,red_bull,Max Verstappen,2.0,3.0,0 days 00:00:00.895000,Finished,18.0,57.0,+00:00.895
63,63,russell,Mercedes,27F4D2,mercedes,George Russell,3.0,4.0,0 days 00:00:08.481000,Finished,15.0,57.0,+00:08.481
12,12,antonelli,Mercedes,27F4D2,mercedes,Andrea Kimi Antonelli,4.0,16.0,0 days 00:00:10.135000,Finished,12.0,57.0,+00:10.135
23,23,albon,Williams,64C4FF,williams,Alexander Albon,5.0,6.0,0 days 00:00:12.773000,Finished,10.0,57.0,+00:12.773


In [422]:

from pathlib import Path
filepath = Path(f'{sample_loc}/laps.parquet')
year = filepath.parts[-4]
round_fol = filepath.parts[-3]
round_no = round_fol.split('_')[1]
round_name = round_fol.split('_')[2:]

# race_id
race_id = f"{year}_{int(round_no):02d}"
print(race_id)
# race_name
print("_".join(round_name))
# session_id
session_code = filepath.parts[-2]
session_id = f"{race_id}_{session_code}"
print(session_id)
# driver_id
driver_id = list(result_df['DriverId'])
print(driver_id)


2025_01
Australian_Grand_Prix
2025_01_R
['norris', 'verstappen', 'russell', 'antonelli', 'albon', 'stroll', 'hulkenberg', 'leclerc', 'piastri', 'hamilton', 'gasly', 'tsunoda', 'ocon', 'bearman', 'lawson', 'bortoleto', 'alonso', 'sainz', 'doohan', 'hadjar']


In [ ]:
# Batch processing
# all_laps = []

# for file_path in Path("f1_parquet_data").glob("*/Round_*/*/laps.parquet"):
#     year = file_path.parts[-4]
#     round_folder = file_path.parts[-3]
#     round_no = round_folder.split('_')[1]
#     session_code = file_path.parts[-2]

#     race_id = f"{year}_{int(round_no):02d}"
#     session_id = f"{race_id}_{session_code}"

#     df = pd.read_parquet(file_path)
#     df['race_id'] = race_id
#     df['session_id'] = session_id
#     all_laps.append(df)

# master_laps_df = pd.concat(all_laps, ignore_index=True)

### **Step 4: Outlier Treatment**

Don't just remove the outliers. An Outlier can be Pit Stop Lap, Safety Car Lap, Wet Weather Lap, Crash Lap.
Do not simply remove them. Instead, create flags like is_pit, is_SC, is_red_flag, is_wet